DataLake (Deltalake) + Lakehouse (Deltatables) - using Delta format (parquet+snappy+delta log)
Delta Lake is an open-source storage framework that brings reliability, ACID transactions, and performance to data lakes. It sits on top of Parquet files and is most commonly used with Apache Spark and Databricks.
Delta Lake is built natively on top of Apache Parquet file. will not support ORC
Delta Lake & Deltalakhouse is the Core/Analytical storage layer behind Bronze–Silver–Gold (medallion) architectures.

https://github.com/inceptezwd37we48/iz_we48_databricks_repo/blob/main/delta_learning/delta_lake_lakehouse_operations_1.ipynb

In [0]:
spark.sql(f"create catalog if not exists lakehousecat1")
spark.sql(f"create schema if not exists lakehousecat1.deltadb")
spark.sql(f"create volume if not exists lakehousecat1.deltadb.deltalake")

In [0]:
df = spark.read.csv("/Volumes/lakehousecat1/deltadb/deltalake/drugsinfo.csv",header=True,inferSchema=True)

# saving as parquet
df.write.format("parquet").mode("overwrite").save("/Volumes/lakehousecat1/deltadb/deltalake/targetparquet")
#saving as delta file
df.write.format("delta").mode("overwrite").save("/Volumes/lakehousecat1/deltadb/deltalake/targetdelta")
# by default saving as delta table
df.write.saveAsTable("lakehousecat1.deltadb.drugstbl",mode="overwrite")

spark.sql("select * from lakehousecat1.deltadb.drugstbl").show()



In [0]:
# taking count from file

spark.read.format("parquet").load("/Volumes/lakehousecat1/deltadb/deltalake/targetparquet").count()
spark.read.format("delta").load("/Volumes/lakehousecat1/deltadb/deltalake/targetdelta").count()

In [0]:
%sql

-- description of table

describe formatted lakehousecat1.deltadb.drugstbl;

In [0]:
%sql
 -- execution plan

explain select * from lakehousecat1.deltadb.drugstbl;

In [0]:
# schema evolution

df.write.option("mergeSchema","true").format("delta").mode("overwrite").saveAsTable("lakehousecat1.deltadb.drugstbl")

spark.sql("select * from lakehousecat1.deltadb.drugstbl").show()



2. DML Operations in Delta Tables & Files

We are overcoming the WORM (Write Once Read Many) limitation in Cloud S3/GCS/ADLS or in Distributed storage layers like HDFS
Delta file/table supports WMRM(Write Manay Read Many) operations, using DMLs such as INSERT/DELETE/UPDATE/MERGE
upsert
update + insert + delete

In [0]:
%sql

create or replace table lakehousecat1.deltadb.sampletb (id int,name string)
using delta;

insert into lakehousecat1.deltadb.sampletb(id,name) values(100,'alluser');


-- for timetravel
describe history lakehousecat1.deltadb.sampletb;


insert into lakehousecat1.deltadb.sampletb(id,name) values(102,'alluser1');
insert into lakehousecat1.deltadb.sampletb(id,name) values(102,'alluser2');
insert into lakehousecat1.deltadb.sampletb(id,name) values(103,'alluser3');

describe history lakehousecat1.deltadb.sampletb;

-- to selet the data from a specific version
select * from lakehousecat1.deltadb.sampletb version as of 1;

-- select current version
select * from lakehousecat1.deltadb.sampletb;

-- to restore the table to a specific version
 restore table lakehousecat1.deltadb.sampletb to version as of 1

In [0]:
%sql
-- Merge Operation
create or replace table drugstbltgt_merge as
select * from lakehousecat1.deltadb.drugstbl where Drug_ID <='D010';

select * from drugstbltgt_merge;

drop table drugstbltgt_merge;

-- merge operation

merge into drugstbltgt_merge tgt 
using lakehousecat1.deltadb.drugstbl src
on tgt.Drug_ID=src.Drug_ID
when matched then
    update set tgt.Drug_Name = src.Drug_Name
when not matched then
    insert (Drug_ID,Drug_Name,Generic_Name,Manufacturer,Category,Dosage_mg,Price_USD,Stock,Expiry_Date,Prescription_Required)
    values(src.Drug_ID,src.Drug_Name ,src.Generic_Name,src.Manufacturer,src.Category,src.Dosage_mg,src.Price_USD,src.Stock,src.Expiry_Date,src.Prescription_Required)
when not matched by source    
then delete;

describe history drugstbltgt_merge;

-- timestamp as of time travel
select * from drugstbltgt_merge timestamp as of '2026-07-30T10:42:50.000+00:00'

In [0]:
%sql
select * from drugstbltgt_merge where Drug_ID = 'D001';

describe history drugstbltgt_merge;

update drugstbltgt_merge
set Drug_Name ='DOLO-650'
where Drug_ID = 'D001';
    
select * from drugstbltgt_merge where Drug_ID ='D001';

select * from drugstbltgt_merge version as of 0
where Drug_ID ='D001';

-- restoring table to specified version

restore table drugstbltgt_merge to version as of 0;

select * from drugstbltgt_merge where Drug_ID ='D001';
    




In [0]:
%sql
--CTAS (Create table As Select)
create or replace table drugstbltgt_merge_test as select * from lakehousecat1.deltadb.drugstbl where  Drug_ID <'D009';

select * from drugstbltgt_merge_test

In [0]:
%sql
-- Vaccum
-- displays table properties
show TBLPROPERTIES drugstbltgt_merge_test;

-- to alter the vaccum days follow below

alter table drugstbltgt_merge_test 
set TBLPROPERTIES(
    'delta.deletedFileRetentionDuration' = '100 days',
    'delta.logRetentionDuration' = '100 days'
    );

VACUUM drugstbltgt_merge_test;

create or replace table drugstbltgt_merge_test1 as select * from drugstbltgt_merge_test;

show TBLPROPERTIES drugstbltgt_merge_test1;
    
VACUUM drugstbltgt_merge_test1 retain 24 hours; 

VACUUM drugstbltgt_merge_test1 RETAIN 1 HOURS dry run;



In [0]:
## Optimize

from pyspark.sql.functions import *

df=spark.range(100).\
                    withColumn("ts",current_timestamp()).\
                    withColumn("user",lit("izuser"))
df.show(5,False)

df.repartition(10).write.format("delta").mode("overwrite").save("/Volumes/lakehousecat1/deltadb/deltalake/targetdir_optimize")


In [0]:
%sql
describe history delta.`/Volumes/lakehousecat1/deltadb/deltalake/targetdir_optimize`;

optimize delta.`/Volumes/lakehousecat1/deltadb/deltalake/targetdir_optimize`;


In [0]:
%sql
-- Drop and Undrop

use lakehousecat1.deltadb;

create table dept (did int,dname string);

insert into dept values(1,'HR'),(2,'IT'),(3,'FINANCE');
insert into dept values(100,'SALES'),(102,'LOANS'),(103,'ADMIN');

select * from dept;

desc history dept;  

drop table dept;

show tables dropped in lakehousecat1.deltadb;

undrop table dept;

select * from dept;

-- Time travel

select * from dept version as of 1;